# SyNF — Symbolic Neural Forecasting (San Juan weekly cases)

Trains the Equation Learner (EQL) on weekly dengue cases with lag 4. Last 52 weeks are held out for testing.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from darts import TimeSeries
from darts.metrics import metrics

from src.config import DATA_DIR, ensure_results_dir
from src.data_utils import series_to_supervised, str_to_function
from src.synf.model import train_synf


In [ ]:
# Load data and configure train/test split
lags = 4
run_name = "synf_sanjuan_lag4"

df = pd.read_csv(DATA_DIR / "Sanjuan_data_weekly.csv")
print(df.head())

col_index = 3  # Cases column
dat = np.array(df.iloc[:, col_index]).reshape(-1, 1)
split_point = dat.shape[0] - 52

tr_dat, ts_dat = dat[:split_point], dat[split_point:]
fore_hor = len(ts_dat)

train_series = TimeSeries.from_values(tr_dat.astype(np.float32))
test_series = TimeSeries.from_values(ts_dat.astype(np.float32))


In [ ]:
# Supervised training matrix
tr_data = series_to_supervised(tr_dat, n_in=lags, n_out=1)
train_data = tr_data.reset_index(drop=True)

X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]

print("Training samples:", len(X_train))
print(X_train.head())


In [ ]:
# Train SyNF (EQL)
func_val, elapsed = train_synf(
    X_train,
    y_train,
    n_lags=lags,
    units=["id", "mul", "cos", "sin", "div"],
    reg=1e-4,
    mask_thresh=0.05,
    iterations=1000,
    run_name=run_name,
)

print("Discovered equation:")
print(func_val)
print(f"Training time: {elapsed:.2f} s")


In [ ]:
# Evaluate discovered equation
math_func = str_to_function(func_val, lags)

train_pred = [math_func(*np.array(X_train.iloc[i])) for i in range(len(X_train))]

plt.figure()
plt.plot(train_pred, label="train_pred")
plt.plot(y_train.values.flatten(), label="y_train")
plt.legend()
plt.title("SyNF training fit")
plt.show()


In [ ]:
# One-step-ahead test predictions
full_data = series_to_supervised(dat, n_in=lags, n_out=1)
test_data = full_data.tail(len(ts_dat)).reset_index(drop=True)

X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]

model_name = "SyNF-Div-Reg"
test_pred_one = [math_func(*np.array(X_test.iloc[i])) for i in range(len(X_test))]

smape = metrics.smape(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(np.array(test_pred_one)),
)
mae = metrics.mae(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(np.array(test_pred_one)),
)
rmse = metrics.rmse(
    TimeSeries.from_values(np.array(y_test)),
    TimeSeries.from_values(np.array(test_pred_one)),
)

print(f"SMAPE: {smape:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"RMSE:  {rmse:.4f}")

plt.figure()
plt.plot(test_pred_one, label="test_pred")
plt.plot(y_test.values.flatten(), label="y_test")
plt.legend()
plt.title("SyNF test fit (last 52 weeks)")
plt.show()


In [ ]:
# Save test predictions
out_dir = ensure_results_dir(run_name)
pred_path = out_dir / f"{run_name}_predictions.csv"
pd.DataFrame({"y_test": y_test.values, "y_pred": test_pred_one}).to_csv(pred_path, index=False)
print(f"Saved predictions to {pred_path}")
